# Collect Data

In [37]:
%pip install pandas numpy seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [38]:
import pandas as pd
import numpy as np

In [39]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [40]:
df = pd.read_csv('final_data.csv')
df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS
0,4073,10597.4769,4800.2166,56.5923,1355.916,744,33.3,2.02232,40592.3314
1,3702,3245.1189,8032.5534,16.1271,386.394,744,33.3,2.02232,40592.3314
2,3563,0.0000,0.0000,0.0000,0.000,0,33.3,2.02232,40592.3314
3,4128,2116.3271,5439.4737,25.6074,613.536,714,33.3,2.02232,40592.3314
4,3609,1978.3072,5187.5167,16.1911,387.928,712,33.3,2.02232,40592.3314
...,...,...,...,...,...,...,...,...,...
21454,4742,1374.4294,32.1202,0.9751,25.130,741,19.2,2.06677,40529.9208
21455,4722,0.0000,0.0000,0.0000,0.000,0,19.2,2.06677,40529.9208
21456,4609,979.8785,136.3452,1.3825,35.633,736,19.2,2.06677,40529.9208
21457,4639,2460.3767,30.4620,16.0628,413.996,742,19.2,2.06677,40529.9208


In [41]:
from math import log

In [42]:
df['Agua_log'] = df['Agua'].apply(lambda x: log(x) if x > 0 else 0)
df['Gas_log'] = df['Gas'].apply(lambda x: log(x) if x > 0 else 0)
df['Idade2'] = df['Idade'].apply(lambda x: x**2)
df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Agua_log,Gas_log,Idade2
0,4073,10597.4769,4800.2166,56.5923,1355.916,744,33.3,2.02232,40592.3314,8.476416,4.035873,16589329
1,3702,3245.1189,8032.5534,16.1271,386.394,744,33.3,2.02232,40592.3314,8.991258,2.780501,13704804
2,3563,0.0000,0.0000,0.0000,0.000,0,33.3,2.02232,40592.3314,0.000000,0.000000,12694969
3,4128,2116.3271,5439.4737,25.6074,613.536,714,33.3,2.02232,40592.3314,8.601438,3.242881,17040384
4,3609,1978.3072,5187.5167,16.1911,387.928,712,33.3,2.02232,40592.3314,8.554010,2.784462,13024881
...,...,...,...,...,...,...,...,...,...,...,...,...
21454,4742,1374.4294,32.1202,0.9751,25.130,741,19.2,2.06677,40529.9208,3.469485,-0.025215,22486564
21455,4722,0.0000,0.0000,0.0000,0.000,0,19.2,2.06677,40529.9208,0.000000,0.000000,22297284
21456,4609,979.8785,136.3452,1.3825,35.633,736,19.2,2.06677,40529.9208,4.915190,0.323893,21242881
21457,4639,2460.3767,30.4620,16.0628,413.996,742,19.2,2.06677,40529.9208,3.416480,2.776506,21520321


In [43]:
target_column = 'Petroleo'
ignore_columns = [
    #'Gas',
    # 'Agua',
    # 'Idade'
]

In [44]:
sample_df = df.sample(frac=1, random_state=42)
sample_df.drop_duplicates(inplace=True)
for col in sample_df.columns:
    Q1 = sample_df[col].quantile(0.25)
    Q3 = sample_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    sample_df = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)]

sample_df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Agua_log,Gas_log,Idade2
5527,5636,508.8115,5839.3422,13.8773,414.564,259,22.20,2.03282,39249.2117,8.672373,2.630254,31764496
15902,4453,102.7398,5.0067,3.7128,54.700,676,30.20,2.00144,38798.0539,1.610777,1.311786,19829209
12124,488,7.6646,0.3782,0.2990,4.247,62,30.20,2.00144,38798.0539,-0.972332,-1.207312,238144
1297,6083,125.6144,564.0495,0.2412,7.476,727,30.23,2.12184,36715.5245,6.335142,-1.422129,37002889
2554,3002,69.0662,0.0276,3.0274,22.671,490,28.70,2.04092,36555.2570,-3.589940,1.107704,9012004
...,...,...,...,...,...,...,...,...,...,...,...,...
6265,5463,0.0000,0.0000,0.0000,0.000,0,18.10,2.00677,39989.7428,0.000000,0.000000,29844369
11964,488,102.3119,5.0487,3.9909,56.696,62,30.20,2.00144,38798.0539,1.619131,1.384017,238144
5390,348,578.2746,0.5789,9.2298,264.040,61,24.70,2.01939,40444.9634,-0.546626,2.222437,121104
860,2638,132.0420,3.2312,5.2944,76.693,619,30.20,2.00144,38798.0539,1.172854,1.666650,6959044


In [45]:
X = sample_df.drop(ignore_columns + [target_column], axis=1)
y = sample_df[target_column]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

In [46]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LinearRegression

In [47]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [48]:
print("Regressão Linear")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, model.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, model.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, model.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, model.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, model.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, model.predict(X_test_scaled)):.4f} (Teste)")

Regressão Linear
RMSE: 833.7957 (Treino), 789.1246 (Teste)
MAE: 510.3713 (Treino), 479.7244 (Teste)
R2: 0.5856 (Treino), 0.5720 (Teste)


# Ridge

In [49]:
alphas = np.logspace(-4, 2, 100)

scoring_metric = 'r2'
ridge_model = Ridge(random_state=42)
grid_search_ridge = GridSearchCV(
    estimator=ridge_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

# 2.2 Treinar o GridSearch
grid_search_ridge.fit(X_train_scaled, y_train)

# Obter o melhor modelo e alpha
best_alpha_ridge = grid_search_ridge.best_params_['alpha']
best_ridge_model = grid_search_ridge.best_estimator_

In [50]:
print("Ridge")
print(f"Best alpha: {best_alpha_ridge}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, best_ridge_model.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, best_ridge_model.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, best_ridge_model.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, best_ridge_model.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, best_ridge_model.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, best_ridge_model.predict(X_test_scaled)):.4f} (Teste)")

Ridge
Best alpha: 5.336699231206313
RMSE: 833.8015 (Treino), 788.9564 (Teste)
MAE: 510.1656 (Treino), 479.5173 (Teste)
R2: 0.5856 (Treino), 0.5722 (Teste)


# Lasso

In [51]:
lasso_model = Lasso(random_state=42, max_iter=2000) 
grid_search_lasso = GridSearchCV(
    estimator=lasso_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_lasso.fit(X_train_scaled, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['alpha']
best_lasso_model = grid_search_lasso.best_estimator_

In [52]:
print("Lasso")
print(f"Best alpha: {best_alpha_lasso}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, best_lasso_model.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, best_lasso_model.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, best_lasso_model.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, best_lasso_model.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, best_lasso_model.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, best_lasso_model.predict(X_test_scaled)):.4f} (Teste)")

Lasso
Best alpha: 1.1497569953977356
RMSE: 833.8444 (Treino), 788.6907 (Teste)
MAE: 509.3400 (Treino), 478.6498 (Teste)
R2: 0.5856 (Treino), 0.5725 (Teste)


# Random Forest

In [53]:
from sklearn.ensemble import RandomForestRegressor

In [70]:
rf_model = RandomForestRegressor(random_state=42)
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid={
        'n_estimators': [100, 200, 300, 400, 500],
        'max_depth': [None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_rf.fit(X_train_scaled, y_train)

best_n_estimators_rf = grid_search_rf.best_params_['n_estimators']
best_max_depth_rf = grid_search_rf.best_params_['max_depth']
best_min_samples_split_rf = grid_search_rf.best_params_['min_samples_split']
best_min_samples_leaf_rf = grid_search_rf.best_params_['min_samples_leaf']
best_rf_model = grid_search_rf.best_estimator_

In [71]:
print("Random Forest")
print(f"Best n_estimators: {best_n_estimators_rf}, max_depth: {best_max_depth_rf}, min_samples_split: {best_min_samples_split_rf}, min_samples_leaf: {best_min_samples_leaf_rf}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, grid_search_rf.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, grid_search_rf.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, grid_search_rf.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, grid_search_rf.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, grid_search_rf.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, grid_search_rf.predict(X_test_scaled)):.4f} (Teste)")

Random Forest
Best n_estimators: 500, max_depth: None, min_samples_split: 2, min_samples_leaf: 1
RMSE: 134.3570 (Treino), 325.8410 (Teste)
MAE: 58.0623 (Treino), 148.6208 (Teste)
R2: 0.9892 (Treino), 0.9270 (Teste)


# Gradient Boosting

In [56]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [57]:
from xgboost import XGBRegressor

In [58]:
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid={
        'n_estimators': [100, 300, 500],
        'learning_rate': [0.01, 0.1, 0.3],
        'max_depth': [3, 5, 7],
        'subsample': [0.7, 1.0],
    },
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search_xgb.fit(X_train_scaled, y_train)

best_n_estimators_xgb = grid_search_xgb.best_params_['n_estimators']
best_max_depth_xgb = grid_search_xgb.best_params_['max_depth']
best_learning_rate_xgb = grid_search_xgb.best_params_['learning_rate']
best_subsample_xgb = grid_search_xgb.best_params_['subsample']
best_xgb_model = grid_search_xgb.best_estimator_

Fitting 3 folds for each of 54 candidates, totalling 162 fits


In [59]:
print("XGBoost")
print(f"Best n_estimators: {best_n_estimators_xgb}, max_depth: {best_max_depth_xgb}, learning_rate: {best_learning_rate_xgb}, subsample: {best_subsample_xgb}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, grid_search_xgb.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, grid_search_xgb.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, grid_search_xgb.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, grid_search_xgb.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, grid_search_xgb.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, grid_search_xgb.predict(X_test_scaled)):.4f} (Teste)")

XGBoost
Best n_estimators: 500, max_depth: 7, learning_rate: 0.1, subsample: 0.7
RMSE: 33.4545 (Treino), 334.6377 (Teste)
MAE: 22.5222 (Treino), 152.8055 (Teste)
R2: 0.9993 (Treino), 0.9230 (Teste)


# KNN Regression

In [60]:
from sklearn.neighbors import KNeighborsRegressor

In [61]:
knn_model = KNeighborsRegressor()
grid_search_knn = GridSearchCV(
    estimator=knn_model,
    param_grid={
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance'],
        'p': [1, 2]
    },
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_knn.fit(X_train_scaled, y_train)
best_n_neighbors_knn = grid_search_knn.best_params_['n_neighbors']
best_weights_knn = grid_search_knn.best_params_['weights']
best_p_knn = grid_search_knn.best_params_['p']
best_knn_model = grid_search_knn.best_estimator_

In [62]:
print("K-Nearest Neighbors")
print(f"Best n_neighbors: {best_n_neighbors_knn}, weights: {best_weights_knn}, p: {best_p_knn}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, grid_search_knn.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, grid_search_knn.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, grid_search_knn.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, grid_search_knn.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, grid_search_knn.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, grid_search_knn.predict(X_test_scaled)):.4f} (Teste)")

K-Nearest Neighbors
Best n_neighbors: 5, weights: distance, p: 1
RMSE: 0.0000 (Treino), 348.7267 (Teste)
MAE: 0.0000 (Treino), 144.3966 (Teste)
R2: 1.0000 (Treino), 0.9164 (Teste)


# SVR (Suport Vector Regression)

In [63]:
from sklearn.svm import SVR

In [64]:
svr_model = SVR()
grid_search_svr = GridSearchCV(
    estimator=svr_model,
    param_grid={
        'C': [0.1, 1, 10, 100],
        'epsilon': [0.01, 0.1, 1],
        'kernel': ['linear', 'rbf']
    },
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_svr.fit(X_train_scaled, y_train)

best_C_svr = grid_search_svr.best_params_['C']
best_epsilon_svr = grid_search_svr.best_params_['epsilon']
best_kernel_svr = grid_search_svr.best_params_['kernel']
best_svr_model = grid_search_svr.best_estimator_

In [65]:
print("Support Vector Regression")
print(f"Best C: {best_C_svr}, epsilon: {best_epsilon_svr}, kernel: {best_kernel_svr}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, grid_search_svr.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, grid_search_svr.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, grid_search_svr.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, grid_search_svr.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, grid_search_svr.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, grid_search_svr.predict(X_test_scaled)):.4f} (Teste)")

Support Vector Regression
Best C: 100, epsilon: 0.01, kernel: rbf
RMSE: 646.1144 (Treino), 592.5581 (Teste)
MAE: 275.9568 (Treino), 256.9283 (Teste)
R2: 0.7512 (Treino), 0.7587 (Teste)


# Neural Network

In [66]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

In [67]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(random_state=42, max_iter=2000))
])

mlp_param_grid = {
    'mlp__hidden_layer_sizes': [(100, 100), (200, 100), (200, 200), 
                                (100, 100, 100)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__solver': ['adam', 'sgd'],
    'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
    'mlp__learning_rate': ['constant', 'adaptive']
}

grid_search_mlp = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=mlp_param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search_mlp.fit(X_train, y_train)
best_params_mlp = grid_search_mlp.best_params_
best_mlp_model = grid_search_mlp.best_estimator_

Fitting 5 folds for each of 128 candidates, totalling 640 fits


c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
120 fits failed out of a total of 640.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
120 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\PC GAMER\Downloads\Codig

In [69]:
print("MLP Regressor")
print(f"Best parameters: {best_params_mlp}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, grid_search_mlp.predict(X_train))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, grid_search_mlp.predict(X_test))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, grid_search_mlp.predict(X_train)):.4f} (Treino), {mean_absolute_error(y_test, grid_search_mlp.predict(X_test)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, grid_search_mlp.predict(X_train)):.4f} (Treino), {r2_score(y_test, grid_search_mlp.predict(X_test)):.4f} (Teste)")

MLP Regressor
Best parameters: {'mlp__activation': 'tanh', 'mlp__alpha': 0.1, 'mlp__hidden_layer_sizes': (200, 200), 'mlp__learning_rate': 'constant', 'mlp__solver': 'adam'}
RMSE: 94.8516 (Treino), 326.4406 (Teste)
MAE: 41.7225 (Treino), 149.3877 (Teste)
R2: 0.9946 (Treino), 0.9268 (Teste)
